# Script 5_6 — Data screening before model estimation (Supp Tables S17–S19)

Checks on the analytic file before any structural model is fitted, on the
variables the models use and the samples they use.

1. **Coding and range.** Each model variable is checked against its valid
   range (rhythm metrics in [0, 1] or ≥ 0; PHQ-9 in 0–27; sleep 2–12 h;
   education 1–5; sex 0/1). A value outside its range stops the notebook.
2. **Missingness and sample.** Observed count per model variable among
   adults, and the analytic-sample cascade (complete exogenous variables;
   at least one observed dependent variable), as script 6 defines it.
3. **Distributions and extreme values.** Skewness, minimum, maximum, the
   number of observations more than 5 s.d. from the mean, and the share of
   rhythm-metric values at a bound of their range.
4. **Pairwise associations.** Survey-weighted Pearson correlations among
   the model variables, pairwise-complete, among adults with rhythm metrics.
5. **Collinearity.** Variance inflation factors for the predictor set of
   each primary outcome equation — the rhythm metric, both biological
   measures and the six covariates — within each age group, from
   survey-weighted least squares.

No observation is excluded on the basis of these checks. Extreme
biological-age values are examined as a sensitivity analysis (script 16);
non-normality is accommodated by the robust maximum-likelihood estimator of
the structural models.

Reads `data/processed/8_nhanes_analytic.csv`.
Writes, to `result/supplementary/`:
`SuppTableS17_Model_Correlations`, `SuppTableS18_VIF` and
`SuppTableS19_Distributions_Missingness` (`.xlsx` with a Notes sheet, and `.csv`).

## Step 0 — Imports, paths, model variables

In [ ]:
import os, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.options.display.max_columns = 60
pd.options.display.width = 200

def find_root():
    """Locate the project root without assuming where the kernel started."""
    env = os.environ.get('NHANES_SEM_ROOT')
    cands = ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().parents]
    for q in cands:
        if (q / 'data' / 'processed').is_dir() and (q / 'scripts').is_dir():
            return q
    raise RuntimeError(
        'cannot find the project root from ' + str(Path.cwd())
        + ' -- start Jupyter inside the project, or set NHANES_SEM_ROOT')

BASE   = find_root()
PROC   = BASE / 'data' / 'processed'
R_SUPP = BASE / 'result' / 'supplementary'
R_SUPP.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(BASE / 'scripts'))

# Reload project modules so the notebook runs against the code on disk.
import importlib
import wls_utils, nature_palette
for _m in (wls_utils, nature_palette):
    importlib.reload(_m)
from wls_utils import wls_fit
from nature_palette import EXP_ORDER, age_label

# Analytic-file names -> the names used in the Mplus files and the tables.
RENAME = {'RIDAGEYR': 'AGE', 'RIAGENDR': 'SEX', 'BMXBMI': 'BMI', 'DMDEDUC2': 'EDU',
          'Wear_Time': 'WEAR_T', 'SLD010H': 'SLEEP_H',
          'KDM_BA_advance': 'BA_ADV', 'SII_log': 'SII_LOG'}
LABEL = {'RA': 'Relative amplitude', 'IS': 'Interdaily stability',
         'IV': 'Intradaily variability',
         'BA_ADV': 'KDM biological-age advancement, years',
         'SII_LOG': 'log SII',
         'PHQ9': 'PHQ-9 total score',
         'COG_Z': 'Cognitive composite, z score (aged \u226560)',
         'AGE': 'Age, years',
         'SEX': 'Sex (0 = male, 1 = female)',
         'BMI': 'Body mass index, kg/m\u00b2',
         'EDU': 'Education (1\u20135)',
         'WEAR_T': 'Accelerometer wear time, min/day',
         'SLEEP_H': 'Sleep duration, h'}
COVS  = ['AGE', 'SEX', 'BMI', 'EDU', 'WEAR_T', 'SLEEP_H']
ORDER = ['RA', 'IS', 'IV', 'BA_ADV', 'SII_LOG', 'PHQ9', 'COG_Z'] + COVS
CELLS = [('18-59', 18, 60), ('60+', 60, 999)]
print('project root:', BASE)

## Step 1 — Load the analytic file; derive the two outcome scores exactly as scripts 6 and 9 do

In [ ]:
df = pd.read_csv(PROC / '8_nhanes_analytic.csv', low_memory=False).rename(columns=RENAME)
assert np.allclose(df['WTMEC4YR'], df['WTMEC2YR'] / 2.0, equal_nan=True), \
    'stored WTMEC4YR does not match WTMEC2YR / 2 -- the file is stale'
adults = df[df['AGE'] >= 18].copy()

# PHQ-9 total: refusal (7) and don't know (9) set to missing; all nine items required.
dpq = adults[[f'DPQ{i:03d}' for i in [10, 20, 30, 40, 50, 60, 70, 80, 90]]].replace({7: np.nan, 9: np.nan})
adults['PHQ9'] = dpq.sum(axis=1, min_count=9)

# Cognition: raw zeros -> missing, z within the 60+ subsample, composite = mean of the three.
elder = adults['AGE'] >= 60
for raw, z in [('memory_score', 'MEM_Z'), ('fluency_score', 'FLUEN_Z'), ('speed_score', 'SPEED_Z')]:
    s = adults[raw].replace(0, np.nan)
    ref = s[elder].dropna()
    adults[z] = np.where(elder, (s - ref.mean()) / ref.std(), np.nan)
adults['COG_Z'] = adults[['MEM_Z', 'FLUEN_Z', 'SPEED_Z']].mean(axis=1, skipna=False)

rhythm = adults[adults['IV'].notna()]
print(f'adults {len(adults):,}; with rhythm metrics {len(rhythm):,} '
      f'(RA is undefined for {int(rhythm["RA"].isna().sum())} of them)')

## Step 2 — Coding and range checks

A value outside its valid range raises: the pipeline does not continue on a coding error. Unbounded variables are listed with their observed range only.

In [ ]:
RANGES = {'RA': (0, 1), 'IS': (0, 1), 'IV': (0, None), 'BA_ADV': (None, None),
          'SII_LOG': (None, None), 'PHQ9': (0, 27), 'COG_Z': (None, None),
          'AGE': (18, 80), 'SEX': (0, 1), 'BMI': (10, 100), 'EDU': (1, 5),
          'WEAR_T': (600, 1440), 'SLEEP_H': (2, 12)}
rows = []
for v, (lo, hi) in RANGES.items():
    s = adults[v].dropna()
    n_bad = int(((s < lo).sum() if lo is not None else 0) + ((s > hi).sum() if hi is not None else 0))
    rows.append(dict(variable=v, n_observed=len(s), minimum=round(s.min(), 3), maximum=round(s.max(), 3),
                     valid_range=('unbounded' if lo is None and hi is None
                                  else f'{lo} to {hi}' if hi is not None else f'\u2265 {lo}'),
                     n_out_of_range=n_bad))
    assert n_bad == 0, f'{v}: {n_bad} values outside the valid range {lo}-{hi}'
for v in ['SEX', 'EDU']:
    assert set(adults[v].dropna().unique()) <= set(range(0, 6)), f'{v}: non-integer codes'
rng = pd.DataFrame(rows)
print(rng.to_string(index=False))
print('\nall model variables within their valid ranges')

## Step 3 — Missingness and the analytic-sample cascade

The cascade follows script 6: complete exogenous variables (rhythm metric, six covariates, weight) and at least one observed dependent variable, which is the sample the Mplus models retain under full-information maximum likelihood.

In [ ]:
miss = pd.DataFrame({
    'variable': ORDER,
    'label': [LABEL[v] for v in ORDER],
    'n_observed': [int(adults[v].notna().sum()) for v in ORDER],
    'pct_missing': [round(100 * adults[v].isna().mean(), 1) for v in ORDER],
})
# the cognitive battery is administered at 60+, so its missingness is quoted within 60+
i = miss.index[miss.variable == 'COG_Z'][0]
miss.loc[i, 'n_observed']  = int(adults.loc[elder, 'COG_Z'].notna().sum())
miss.loc[i, 'pct_missing'] = round(100 * adults.loc[elder, 'COG_Z'].isna().mean(), 1)

x_ok = adults.dropna(subset=['IV', 'WTMEC4YR'] + COVS)
dep  = x_ok[x_ok[['BA_ADV', 'SII_LOG', 'PHQ9']].notna().any(axis=1)]
cog  = x_ok[(x_ok['AGE'] >= 60)
            & x_ok[['BA_ADV', 'SII_LOG', 'MEM_Z', 'FLUEN_Z', 'SPEED_Z']].notna().any(axis=1)]
cascade = pd.DataFrame([
    ('Adults aged \u226518 years', len(adults)),
    ('with valid rest\u2013activity rhythm metrics', len(rhythm)),
    ('with the six covariates observed', len(x_ok)),
    ('depression models: at least one dependent variable observed', len(dep)),
    ('    aged 18\u201359 years', int((dep['AGE'] < 60).sum())),
    ('    aged \u226560 years', int((dep['AGE'] >= 60).sum())),
    ('cognition models (aged \u226560): at least one dependent variable observed', len(cog)),
], columns=['step', 'n'])
print(miss.to_string(index=False)); print(); print(cascade.to_string(index=False))

## Step 4 — Distributions and extreme values

Unweighted, among adults (the cognitive composite among adults aged 60 and over). The share at a bound uses the theoretical range of each rhythm metric: 0 and 1 for RA and IS, 0 for IV.

In [ ]:
rows = []
for v in ORDER:
    s = (adults.loc[elder, v] if v == 'COG_Z' else adults[v]).dropna()
    z = (s - s.mean()) / s.std()
    r = dict(variable=v, label=LABEL[v], n=len(s), minimum=s.min(), maximum=s.max(),
             mean=s.mean(), sd=s.std(), skewness=float((z ** 3).mean()),
             n_beyond_5sd=int((z.abs() > 5).sum()), pct_at_bound=np.nan)
    if v in ('RA', 'IS'):
        r['pct_at_bound'] = 100 * ((s <= 1e-9) | (s >= 1 - 1e-9)).mean()
    elif v == 'IV':
        r['pct_at_bound'] = 100 * (s <= 1e-9).mean()
    rows.append(r)
dist = pd.DataFrame(rows)
print(dist.round(3).to_string(index=False))

## Step 5 — Survey-weighted pairwise correlations

Pearson correlations weighted by the four-year examination weight, pairwise-complete, among adults with rhythm metrics. Correlations with the cognitive composite are among adults aged 60 and over by construction.

In [ ]:
def weighted_corr(x, y, w):
    m = x.notna() & y.notna() & w.notna() & (w > 0)
    if m.sum() < 30:
        return np.nan
    x, y, w = x[m].to_numpy(float), y[m].to_numpy(float), w[m].to_numpy(float)
    xm, ym = np.average(x, weights=w), np.average(y, weights=w)
    cxy = np.average((x - xm) * (y - ym), weights=w)
    return cxy / np.sqrt(np.average((x - xm) ** 2, weights=w) * np.average((y - ym) ** 2, weights=w))

corr = pd.DataFrame(index=ORDER, columns=ORDER, dtype=float)
for a in ORDER:
    for b in ORDER:
        corr.loc[a, b] = weighted_corr(rhythm[a], rhythm[b], rhythm['WTMEC4YR'])
print(corr.round(2).to_string())

off = corr.where(~np.eye(len(ORDER), dtype=bool))
i_max = off.abs().stack().idxmax()
print(f'\nlargest |r| among model variables: {i_max[0]} x {i_max[1]} = {off.loc[i_max]:.2f}')
print(f'between the two biological measures, BA_ADV x SII_LOG: r = {corr.loc["BA_ADV", "SII_LOG"]:.3f}')

## Step 6 — Variance inflation factors

For each primary outcome equation the predictors are the rhythm metric, both biological measures and the six covariates. VIF_j = 1 / (1 − R²_j), where R²_j is from the survey-weighted least-squares regression of predictor j on the other predictors, within the age group, on participants with every predictor observed.

In [ ]:
def vif_set(frame, exposure):
    preds = [exposure, 'BA_ADV', 'SII_LOG'] + COVS
    s = frame.dropna(subset=preds + ['WTMEC4YR'])
    s = s[s['WTMEC4YR'] > 0]
    w = s['WTMEC4YR'].to_numpy(float)
    out = {}
    for j in preds:
        others = [p for p in preds if p != j]
        r2 = wls_fit(s[others].to_numpy(float), s[j].to_numpy(float), w=w)['r2']
        out[j] = 1.0 / (1.0 - r2)
    return len(s), out

rows = []
for cell_, lo, hi in CELLS:
    frame = adults[(adults['AGE'] >= lo) & (adults['AGE'] < hi)]
    for exp in EXP_ORDER:
        n, v = vif_set(frame, exp)
        for j, val in v.items():
            rows.append(dict(age_group=age_label(cell_), exposure=exp, n=n,
                             predictor='rhythm metric' if j == exp else j, VIF=val))
vif = pd.DataFrame(rows)
vif_wide = (vif.pivot_table(index=['age_group', 'exposure', 'n'], columns='predictor', values='VIF')
            .reset_index())
vif_wide.columns.name = None
vif_wide = vif_wide[['age_group', 'exposure', 'n', 'rhythm metric', 'BA_ADV', 'SII_LOG'] + COVS]
print(vif_wide.round(2).to_string(index=False))
print(f'\nlargest VIF over the six specifications: {vif.VIF.max():.2f}')
assert vif.VIF.max() < 5, 'a predictor exceeds VIF 5'

## Step 7 — Write Supplementary Tables S17, S18 and S19

In [ ]:
def write_table(stem, sheet, frame, notes):
    out = R_SUPP / f'{stem}.xlsx'
    with pd.ExcelWriter(out, engine='openpyxl') as xw:
        frame.to_excel(xw, sheet_name=sheet, index=False)
        pd.DataFrame({'Notes': notes}).to_excel(xw, sheet_name='Notes', index=False)
        for ws in xw.book.worksheets:
            for col in ws.columns:
                width = max((len(str(c.value)) for c in col if c.value is not None), default=8)
                ws.column_dimensions[col[0].column_letter].width = min(max(width + 2, 8), 48)
            ws.freeze_panes = 'A2'
    frame.to_csv(out.with_suffix('.csv'), index=False)
    print(f'wrote {out.relative_to(BASE)} and .csv')

ABBREV = ('BA_ADV, KDM biological-age advancement; KDM, Klemera-Doubal method; '
          'PHQ-9, 9-item Patient Health Questionnaire; SD, standard deviation; '
          'SII, systemic immune-inflammation index; VIF, variance inflation factor.')

# S17: correlation matrix, lower triangle with the diagonal
s17 = corr.round(2).copy()
s17.index = [LABEL[v] for v in ORDER]
s17 = s17.where(np.tril(np.ones(s17.shape, dtype=bool)))
s17.insert(0, 'Variable', s17.index)
s17 = s17.reset_index(drop=True)
write_table('SuppTableS17_Model_Correlations', 'Supp Table S17', s17, [
    'Supplementary Table S17 | Survey-weighted pairwise correlations among the '
    'model variables.',
    'Pearson correlations weighted by the four-year mobile examination centre '
    f'weight, pairwise-complete, among the {len(rhythm):,} adults with '
    'rest-activity rhythm metrics; lower triangle shown. Correlations with the '
    'cognitive composite are among adults aged 60 years and over, to whom the '
    'cognitive battery is administered. Sex is coded 0 = male, 1 = female; '
    'education 1 (less than 9th grade) to 5 (college graduate).',
    f'The largest correlation among the model variables is between {LABEL[i_max[0]].lower()} '
    f'and {LABEL[i_max[1]].lower()} (r = {off.loc[i_max]:.2f}); the two are never entered in '
    'the same model. The two biological measures correlate at '
    f'r = {corr.loc["BA_ADV", "SII_LOG"]:.2f}.',
    'Computed before any model was fitted (script 5_6); no observation is '
    'excluded on the basis of this table.',
    ABBREV,
])

# S18: VIF per outcome-equation predictor set
s18 = vif_wide.round(2).rename(columns={
    'age_group': 'Age group', 'exposure': 'Rhythm metric', 'n': 'n',
    'rhythm metric': 'Rhythm metric (VIF)'})
write_table('SuppTableS18_VIF', 'Supp Table S18', s18, [
    'Supplementary Table S18 | Variance inflation factors for the predictor '
    'set of each primary outcome equation.',
    'For each rhythm metric and age group, the predictors of the outcome '
    'equation of the primary structural model: the rhythm metric, KDM '
    'biological-age advancement, log SII and the six covariates. VIF = '
    '1 / (1 - R-squared) from the survey-weighted least-squares regression of '
    'each predictor on the others, on participants with every predictor '
    'observed (n). The mediator equations use subsets of the same predictors.',
    f'The largest value is {vif.VIF.max():.2f}.',
    'Computed before any model was fitted (script 5_6).',
    ABBREV,
])

# S19: distributions, extreme values, missingness
s19 = dist.merge(miss[['variable', 'n_observed', 'pct_missing']], on='variable')
s19 = s19[['label', 'n', 'minimum', 'maximum', 'mean', 'sd', 'skewness',
           'n_beyond_5sd', 'pct_at_bound', 'pct_missing']]
s19.columns = ['Variable', 'n', 'Minimum', 'Maximum', 'Mean', 'SD', 'Skewness',
               'n beyond 5 SD', '% at a bound', '% missing among adults']
for c in ['Minimum', 'Maximum', 'Mean', 'SD', 'Skewness', '% at a bound']:
    s19[c] = s19[c].round(2)
write_table('SuppTableS19_Distributions_Missingness', 'Supp Table S19', s19, [
    'Supplementary Table S19 | Distributions, extreme values and missingness '
    'of the model variables.',
    f'Unweighted summaries among adults aged 18 years and over (n = {len(adults):,}); '
    'the cognitive composite among adults aged 60 and over. Skewness is the '
    'standardised third moment. "% at a bound" is the share of rhythm-metric '
    'values at a limit of the metric\'s range (0 or 1 for relative amplitude '
    'and interdaily stability, 0 for intradaily variability). Missingness is '
    'relative to all adults (for the cognitive composite, to adults aged 60 '
    'and over).',
    'Every model variable lies within its valid range. No observation is '
    'excluded on the basis of this table: extreme values of biological age '
    'are retained and examined in Supplementary Table S11, and departures '
    'from normality are accommodated by the robust maximum-likelihood '
    'estimator of the structural models.',
    'Analytic-sample cascade (unweighted): '
    + '; '.join(f'{s.strip()}, n = {n:,}' for s, n in cascade.values) + '.',
    'Computed before any model was fitted (script 5_6).',
    ABBREV,
])

## Step 8 — Summary

In [ ]:
print(f'range checks           : all {len(RANGES)} model variables within their valid ranges')
print(f'covariate-complete     : {len(x_ok):,} of {len(rhythm):,} adults with rhythm metrics')
print(f'depression / cognition : {len(dep):,} / {len(cog):,} enter the structural models')
print(f'largest |r|            : {i_max[0]} x {i_max[1]} = {off.loc[i_max]:.2f}')
print(f'BA_ADV x log SII       : r = {corr.loc["BA_ADV", "SII_LOG"]:.3f}')
print(f'largest VIF            : {vif.VIF.max():.2f}')
print('beyond 5 SD            : ' + ', '.join(f'{r.variable} {r.n_beyond_5sd}' for r in dist.itertuples() if r.n_beyond_5sd))
print('at a bound             : ' + ', '.join(f'{r.variable} {r.pct_at_bound:.2f}%' for r in dist.itertuples() if r.variable in ('RA', 'IS', 'IV')))